In [ ]:
import jax
import flax.linen as nn
import jax.numpy as jnp

import jVMC_exp.global_defs as global_defs
import jVMC_exp.nets.activation_functions as act_funs
from jVMC_exp.nets.initializers import init_fn_args

import jVMC_exp.nets.initializers

def log_cosh(x): # FIXED 
    two = jnp.asarray(2, dtype=x.real.dtype)
    half_log_two = jnp.log(two)

    sgn_x = -two * jnp.signbit(x.real) + 1
    sgn_x = sgn_x.astype(x.dtype)

    x = x * sgn_x

    return x + jnp.log1p(jnp.exp(-two * x)) - half_log_two

# def log_cosh(x):
#     sgn_x = -2 * jnp.signbit(x.real) + 1
#     x = x * sgn_x
#     return x + jnp.log1p(jnp.exp(-2.0 * x)) - jnp.log(jnp.asarray(2, dtype=x.real.dtype))

class CpxRBM(nn.Module):
    """Restricted Boltzmann machine with complex parameters.

    Initialization arguments:
        * ``s``: Computational basis configuration.
        * ``numHidden``: Number of hidden units.
        * ``bias``: ``Boolean`` indicating whether to use bias.

    """
    numHidden: int = 2
    bias: bool = False
    param_dtype: jnp.dtype = jnp.complex128

    @nn.compact
    def __call__(self, s):

        layer = nn.Dense(self.numHidden, use_bias=self.bias,
                         **init_fn_args(kernel_init=jVMC_exp.nets.initializers.cplx_init,
                                        bias_init=jax.nn.initializers.zeros,
                                        param_dtype=self.param_dtype),
                         )
        
        x = log_cosh(layer(2 * s.ravel() - 1)) # Needed to infer dtype

        return jnp.sum(x, dtype=x.dtype)

In [103]:
import copy

allowed_dtyes_real = {
    "f64": jnp.float64,
    "f32": jnp.float32,
    "f16": jnp.float16,
    "bf16": jnp.bfloat16
}

allowed_dtyes_cpx = {
    "c128": jnp.complex128,
    "c64": jnp.complex64
}

a = copy.deepcopy(allowed_dtyes_cpx)
a.update(allowed_dtyes_real)

allowed_dtyes_cpx, a, allowed_dtyes_real

({'c128': jax.numpy.complex128, 'c64': jax.numpy.complex64},
 {'c128': jax.numpy.complex128,
  'c64': jax.numpy.complex64,
  'f64': jax.numpy.float64,
  'f32': jax.numpy.float32,
  'f16': jax.numpy.float16,
  'bf16': jax.numpy.bfloat16},
 {'f64': jax.numpy.float64,
  'f32': jax.numpy.float32,
  'f16': jax.numpy.float16,
  'bf16': jax.numpy.bfloat16})

In [91]:
input = jnp.ones((10), dtype=jnp.int32)

model = CpxRBM(2, True, param_dtype=jnp.complex128)

params = model.init(jax.random.PRNGKey(123), input)

In [92]:
model.apply(params, input)

Array(-0.00108655+0.00467096j, dtype=complex128)

In [93]:
params

{'params': {'Dense_0': {'kernel': Array([[0.00525831+4.51924325e-03j, 0.0092928 +6.06010755e-03j],
          [0.00455027+1.13953645e-03j, 0.00021895+7.62392978e-03j],
          [0.00758064+1.49825675e-03j, 0.00576009+5.87359597e-03j],
          [0.00855581+4.70576448e-03j, 0.00412297+2.24175583e-03j],
          [0.00189032+9.52613780e-03j, 0.00350568+7.71964269e-05j],
          [0.00539296+7.09100086e-03j, 0.00063165+2.42390373e-03j],
          [0.00145957+8.88137776e-03j, 0.00063186+1.20674451e-03j],
          [0.00561433+8.91859382e-03j, 0.00544056+3.25528500e-03j],
          [0.00058855+8.05178097e-03j, 0.00370987+8.77146496e-03j],
          [0.00990292+9.58601124e-03j, 0.00021436+4.88920102e-03j]],      dtype=complex128),
   'bias': Array([0.+0.j, 0.+0.j], dtype=complex128)}}}

In [94]:
params_64 = jax.tree_util.tree_map(lambda x: x.astype(jnp.float16), params)

In [95]:
jax.make_jaxpr(model.apply)(params_64, input)

{ lambda ; a:f16[2] b:f16[10,2] c:i32[10]. let
    d:i32[10] = mul 2:i32[] c
    e:i32[10] = sub d 1:i32[]
    f:f16[10] = convert_element_type[new_dtype=float16 weak_type=False] e
    g:f16[2] = dot_general[dimension_numbers=(([0], [0]), ([], []))] f b
    h:f16[2] = add g a
    i:bool[2] = jit[
      name=signbit
      jaxpr={ lambda ; h:f16[2]. let
          j:i16[2] = bitcast_convert_type[new_dtype=int16] h
          k:i16[2] = shift_right_arithmetic j 15:i16[]
          i:bool[2] = convert_element_type[new_dtype=bool weak_type=False] k
        in (i,) }
    ] h
    l:i64[2] = convert_element_type[new_dtype=int64 weak_type=True] i
    m:i64[2] = mul -2:i64[] l
    n:i64[2] = add m 1:i64[]
    o:f16[2] = convert_element_type[new_dtype=float16 weak_type=False] n
    p:f16[2] = mul h o
    q:f16[2] = mul -2.0:f16[] p
    r:f16[2] = exp q
    s:f16[2] = log1p r
    t:f16[2] = add p s
    u:f16[] = log 2.0:f16[]
    v:f16[2] = sub t u
    w:f16[] = reduce_sum[axes=(0,) out_sharding=None